In [207]:
import pandas as pd
import numpy as np
import openpyxl

In [208]:
df_sales = pd.read_csv('./sales.csv', sep=',')
df_stores = pd.read_csv('./stores.csv', sep=',')

In [209]:
df_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   store_id  31 non-null     str    
 1   date      31 non-null     str    
 2   category  31 non-null     str    
 3   product   31 non-null     str    
 4   quantity  31 non-null     int64  
 5   price     31 non-null     int64  
 6   discount  31 non-null     float64
 7   is_promo  31 non-null     int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 2.1 KB


In [210]:
df_sales['date'] = pd.to_datetime(df_sales['date'])

df_sales['year'] = df_sales['date'].dt.year
df_sales['month'] = df_sales['date'].dt.month
df_sales['quarter'] = df_sales['date'].dt.quarter
df_sales['revenue'] = ( df_sales['quantity'] * df_sales['price'] * (1 - df_sales['discount']) ).round(2)

df_sales.head()

,store_id,date,category,product,quantity,price,discount,is_promo,year,month,quarter,revenue
0,S001,2024-01-15,Электроника,Ноутбук,2,50000,0.05,1,2024,1,1,95000.0
1,S001,2024-01-15,Электроника,Мышь,5,1500,0.10,1,2024,1,1,6750.0
2,S001,2024-01-15,Бытовая техника,Чайник,3,3000,0.00,0,2024,1,1,9000.0
3,S002,2024-01-15,Электроника,Ноутбук,1,52000,0.05,1,2024,1,1,49400.0
4,S002,2024-01-15,Бытовая техника,Чайник,2,3100,0.00,0,2024,1,1,6200.0


In [211]:
df_sales.isna().sum()

store_id    0
date        0
category    0
product     0
quantity    0
price       0
discount    0
is_promo    0
year        0
month       0
quarter     0
revenue     0
dtype: int64

In [212]:
count_duples_sales = df_sales.duplicated().sum()
count_duples_sales

np.int64(0)

In [213]:
df_sales[df_sales.duplicated(subset=['store_id', 'date', 'category', 'product'])]

,store_id,date,category,product,quantity,price,discount,is_promo,year,month,quarter,revenue


In [214]:
df_stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   store_id    3 non-null      str  
 1   store_name  3 non-null      str  
 2   city        3 non-null      str  
 3   region      3 non-null      str  
dtypes: str(4)
memory usage: 228.0 bytes


In [215]:
df_stores.isna().sum()

store_id      0
store_name    0
city          0
region        0
dtype: int64

In [216]:
count_duples_stores = df_stores.duplicated().sum()

count_duples_stores

np.int64(0)

In [217]:
df_stores

,store_id,store_name,city,region
0,S001,Магазин на Ленина,Москва,Центр
1,S002,Торговый центр Запад,СПб,Северо-Запад
2,S003,Гипермаркет Восток,Казань,Приволжье


In [218]:
df_stores.drop_duplicates()

,store_id,store_name,city,region
0,S001,Магазин на Ленина,Москва,Центр
1,S002,Торговый центр Запад,СПб,Северо-Запад
2,S003,Гипермаркет Восток,Казань,Приволжье


In [219]:
df_sales_stores = pd.merge(df_sales,df_stores,on='store_id',how='left')

df_sales_stores.head()

,store_id,date,category,product,quantity,price,discount,is_promo,year,month,quarter,revenue,store_name,city,region
0,S001,2024-01-15,Электроника,Ноутбук,2,50000,0.05,1,2024,1,1,95000.0,Магазин на Ленина,Москва,Центр
1,S001,2024-01-15,Электроника,Мышь,5,1500,0.10,1,2024,1,1,6750.0,Магазин на Ленина,Москва,Центр
2,S001,2024-01-15,Бытовая техника,Чайник,3,3000,0.00,0,2024,1,1,9000.0,Магазин на Ленина,Москва,Центр
3,S002,2024-01-15,Электроника,Ноутбук,1,52000,0.05,1,2024,1,1,49400.0,Торговый центр Запад,СПб,Северо-Запад
4,S002,2024-01-15,Бытовая техника,Чайник,2,3100,0.00,0,2024,1,1,6200.0,Торговый центр Запад,СПб,Северо-Запад


In [220]:
df_total_revenue_store_cities = df_sales_stores.groupby(by=['store_id', 'city']).agg(
    total_revenue=('revenue','sum')

).sort_values('total_revenue', ascending=False).reset_index()

df_total_revenue_store_cities

,store_id,city,total_revenue
0,S001,Москва,662690.0
1,S003,Казань,239260.0
2,S002,СПб,222292.0


In [221]:
df_temp = df_sales_stores.groupby(by=['category','product']).agg(
    total_revenue=('revenue', 'sum')
).sort_values('total_revenue', ascending=False).reset_index()

df_temp['rank'] = df_temp.groupby('category')['total_revenue'].rank(method='dense', ascending=False)

df_top3 = df_temp[df_temp['rank'] <= 3]

df_top3

,category,product,total_revenue,rank
0,Электроника,Ноутбук,911075.0,1.0
1,Электроника,Мышь,100382.0,2.0
2,Бытовая техника,Чайник,68760.0,1.0
3,Бытовая техника,Утюг,44025.0,2.0


In [222]:
df_month_revenue = df_sales_stores.groupby(by='month').agg(
    total_revenue=('revenue', 'sum')
).sort_values('month').reset_index()

df_month_revenue

,month,total_revenue
0,1,166350.0
1,2,70400.0
2,3,263050.0
3,4,52920.0
4,5,23790.0
5,6,115527.0
6,7,78620.0
7,8,122445.0
8,9,81850.0
9,10,149290.0


In [223]:
df_month_revenue['ma_3'] = ( df_month_revenue['total_revenue'].rolling(window=3, min_periods=1).mean() ).round(2)

df_month_revenue

,month,total_revenue,ma_3
0,1,166350.0,166350.00
1,2,70400.0,118375.00
2,3,263050.0,166600.00
3,4,52920.0,128790.00
4,5,23790.0,113253.33
5,6,115527.0,64079.00
6,7,78620.0,72645.67
7,8,122445.0,105530.67
8,9,81850.0,94305.00
9,10,149290.0,117861.67


In [224]:
df_promo = df_sales_stores[df_sales_stores['is_promo'] == 1]

df_month_promo = df_promo.groupby(by='month').agg(
    promo_revenue=('revenue','sum')
).sort_values('month').reset_index()

df_month_promo

,month,promo_revenue
0,1,151150.0
1,2,21400.0
2,3,251450.0
3,4,52920.0
4,5,23790.0
5,6,102777.0
6,7,25620.0
7,8,122445.0
8,9,67650.0
9,10,149290.0


In [225]:
df_total_promo = pd.merge(df_month_revenue,df_month_promo,on='month',how='left')

df_total_promo['promo_share'] = np.where(df_total_promo['promo_revenue'] != 0, (df_total_promo['promo_revenue'] / df_total_promo['total_revenue']) * 100, 0).round(2)

df_total_promo[['month','total_revenue','promo_revenue','promo_share']]

,month,total_revenue,promo_revenue,promo_share
0,1,166350.0,151150.0,90.86
1,2,70400.0,21400.0,30.40
2,3,263050.0,251450.0,95.59
3,4,52920.0,52920.0,100.00
4,5,23790.0,23790.0,100.00
5,6,115527.0,102777.0,88.96
6,7,78620.0,25620.0,32.59
7,8,122445.0,122445.0,100.00
8,9,81850.0,67650.0,82.65
9,10,149290.0,149290.0,100.00


In [226]:
df_monthly_store = df_sales_stores.groupby(by=['store_id','month']).agg(
    total_revenue=('revenue','sum')
).reset_index()

df_monthly_store = df_monthly_store.sort_values(['store_id','month'])

df_monthly_store['pct_change'] = df_monthly_store.groupby(['store_id'])['total_revenue'].pct_change() * 100

df_monthly_store['pct_change'] = df_monthly_store['pct_change'].fillna(0)

df_monthly_store['pct_change'] = df_monthly_store['pct_change'].round(2)

df_monthly_store.head()

,store_id,month,total_revenue,pct_change
0,S001,1,110750.0,0.00
1,S001,2,60900.0,-45.01
2,S001,3,150750.0,147.54
3,S001,5,12960.0,-91.40
4,S001,6,93600.0,622.22


In [227]:
df_cat_store = df_sales_stores.groupby(by=['store_id','category'])['revenue'].sum().reset_index()

df_store_total = df_sales_stores.groupby(by='store_id')['revenue'].sum().reset_index()

df_cat_store = df_cat_store.rename(columns={'revenue':'cat_store_revenue'})

df_store_total = df_store_total.rename(columns={'revenue':'store_total_revenue'})

df_cat_store = pd.merge(df_cat_store,df_store_total,on='store_id',how='left')

df_cat_store['cat_share'] = ( df_cat_store['cat_store_revenue'] / df_cat_store['store_total_revenue'] * 100 ).round(2)

df_cat_store[['store_id','category','cat_share']]

,store_id,category,cat_share
0,S001,Бытовая техника,7.81
1,S001,Электроника,92.19
2,S002,Бытовая техника,16.86
3,S002,Электроника,83.14
4,S003,Бытовая техника,9.86
5,S003,Электроника,90.14


In [112]:
df_avg_monthly = df_monthly_store.groupby(by='store_id').agg(
    avg_revenue=('total_revenue','mean')
).reset_index()

df_avg_monthly['avg_revenue'] = df_avg_monthly['avg_revenue'].round(2)

df_season = pd.merge(df_monthly_store,df_avg_monthly,on='store_id',how='left')

df_season['seasonality'] = ( df_season['total_revenue'] / df_season['avg_revenue']  ).round(2)

df_season['seasonality_persentege'] = ( (df_season['seasonality'] - 1) * 100)

df_season

,store_id,month,total_revenue,pct_change,avg_revenue,seasonality,seasonality_persentege
0,S001,1,110750.0,0.00,73632.22,1.50,50.0
1,S001,2,60900.0,-45.01,73632.22,0.83,-17.0
2,S001,3,150750.0,147.54,73632.22,2.05,105.0
3,S001,5,12960.0,-91.40,73632.22,0.18,-82.0
4,S001,6,93600.0,622.22,73632.22,1.27,27.0
5,S001,7,67280.0,-28.12,73632.22,0.91,-9.0
6,S001,8,14625.0,-78.26,73632.22,0.20,-80.0
7,S001,9,15400.0,5.30,73632.22,0.21,-79.0
8,S001,10,136425.0,785.88,73632.22,1.85,85.0
9,S002,1,55600.0,0.00,24699.11,2.25,125.0


In [116]:
with pd.ExcelWriter('result.xlsx') as writer:
    df_monthly_store.to_excel(writer,sheet_name='pct_change_by_store', index=False)
    df_cat_store.to_excel(writer,sheet_name='cat_share_by_store',index=False)
    df_season.to_excel(writer,sheet_name='seasons_by_store', index=False)

In [122]:
df_monthly_store_pivot = df_monthly_store.pivot_table(
    values='total_revenue',
    index='store_id',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

df_monthly_store_pivot

month,1,2,3,4,5,6,7,8,9,10
store_id,,,,,,,,,,
S001,110750.0,60900.0,150750.0,0.0,12960.0,93600.0,67280.0,14625.0,15400.0,136425.0
S002,55600.0,9500.0,100700.0,7020.0,0.0,9177.0,11340.0,9720.0,14200.0,5035.0
S003,0.0,0.0,11600.0,45900.0,10830.0,12750.0,0.0,98100.0,52250.0,7830.0


In [228]:
def categorize_product(value):
    if value >= 200_000:
        return 'high'
    elif value > 100_000:
        return 'medium'
    else:
        return 'low'
    

df_product_cat = df_sales_stores.groupby(by='product')['revenue'].sum().reset_index()

df_product_cat = df_product_cat.rename(columns={'revenue':'prdct_revenue'})

df_product_cat['product_category'] = df_product_cat['prdct_revenue'].apply(categorize_product)

df_product_cat = df_product_cat.sort_values('prdct_revenue', ascending=False).reset_index()

df_product_cat = df_product_cat[['product','product_category']]

df_sales_stores = df_sales_stores.merge(df_product_cat,on='product',how='left')

df_sales_stores.head()

,store_id,date,category,product,quantity,price,discount,is_promo,year,month,quarter,revenue,store_name,city,region,product_category
0,S001,2024-01-15,Электроника,Ноутбук,2,50000,0.05,1,2024,1,1,95000.0,Магазин на Ленина,Москва,Центр,high
1,S001,2024-01-15,Электроника,Мышь,5,1500,0.10,1,2024,1,1,6750.0,Магазин на Ленина,Москва,Центр,medium
2,S001,2024-01-15,Бытовая техника,Чайник,3,3000,0.00,0,2024,1,1,9000.0,Магазин на Ленина,Москва,Центр,low
3,S002,2024-01-15,Электроника,Ноутбук,1,52000,0.05,1,2024,1,1,49400.0,Торговый центр Запад,СПб,Северо-Запад,high
4,S002,2024-01-15,Бытовая техника,Чайник,2,3100,0.00,0,2024,1,1,6200.0,Торговый центр Запад,СПб,Северо-Запад,low


In [ ]:
df_sales_stores['revenue_by_category'] = df_sales_stores.groupby('category')['revenue'].transform('sum')

df_sales_stores['category_revenue_share'] = (df_sales_stores['revenue'] / df_sales_stores['revenue_by_category'] * 100 ).round(2)

df_sales_stores = df_sales_stores.drop(['revenue_by_category'], axis=1)

df_electricity = df_sales_stores[df_sales_stores['category'] == 'Электроника']

check_share = df_electricity['category_revenue_share'].sum()

print(check_share)

99.99999999999999
